# Week 13 — BBO capstone driver (final, scored)

Round 13, the last submission. **Scoring is on this round alone**, not on the best value found across the campaign.

The budget goes on perpendicular probes: F4, F7 and F8 have spent line searches with located vertices, so any further gain has to come from off-line moves. F3 abandons its exhausted neighbourhood for a λ=0.7 centre-ward jump. F5 continues its ray to n=11, one step short of the domain floor. F1 chases the lobe centre between estimated nodes.

**The error, stated plainly and only once.** Under final-round-only scoring the correct terminal move is to resubmit each function's best-known coordinates and spend nothing on exploration. I treated this as another exploration round. The audit cell below is what should have been run before choosing, and it is the last cell of the project for a reason.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 13
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 13
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: 'lobe centre between nodes',
    2: 'diagonal vertex + perpendicular offset',
    3: 'abandon neighbourhood, λ=0.7 centre-ward',
    4: 'vertex + perpendicular',
    5: 'n=11 along ray',
    6: 'vertex, x2 dropped 0.15',
    7: 'first ever perpendicular probe',
    8: 'vertex + perpendicular',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 12. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — perpendicular moves off the spent lines

In [ ]:
proposals = {
    1: np.array([0.4875, 0.5289]),
    2: np.array([0.674, 0.722]),
    3: np.array([0.383383, 0.582295, 0.505085]),
    4: np.array([0.430293, 0.404078, 0.540611, 0.589237]),
    5: np.array([0.231806, 0.807874, 0.009498, 0.738917]),
    6: np.array([0.305686, 0.363889, 0.704651, 0.647233, 0.377423]),
    7: np.array([0.643824, 0.372122, 0.593497, 0.459377, 0.528142, 0.445084]),
    8: np.array([0.134496, 0.492931, 0.258165, 0.509239, 0.618801, 0.424815, 0.645029, 0.32563]),
}

pd.DataFrame([dict(func=f"F{fid}", d=bbo.DIMS[fid],
                   submission=bbo.submission(proposals[fid]))
              for fid in bbo.FUNC_IDS])


### The terminal audit — what a lock-in round would have submitted

Under final-round-only scoring, this table is the decision. Every row where `exploring` is True is a function whose score depends on a probe rather than on a value already banked.

In [ ]:
lock = bbo.ledger(up_to=PRIOR)
rows = []
for fid in bbo.FUNC_IDS:
    xb, yb, rb = bbo.best_point(fid, up_to=PRIOR)
    rows.append(dict(func=f"F{fid}", banked=f"{yb:.6g}", from_round=rb,
                     exploring=not np.allclose(xb, proposals[fid], atol=1e-6),
                     lock_in_would_submit=bbo.submission(xb)))
pd.DataFrame(rows)


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.669937, 0.751452],
    2: [0.354432, 0.449899],
    3: [0.111275, 0.774316, 0.516951],
    4: [0.183872, 0.065353, 0.017618, 0.904326],
    5: [0.308806, 0.730874, 0.091998, 0.661917],
    6: [0.070021, 0.530732, 0.952853, 0.825805, 0.228797],
    7: [0.965568, 0.153915, 0.588691, 0.807159, 0.099427, 0.700138],
    8: [0.038983, 0.275485, 0.181927, 0.347803, 0.803194, 0.234763, 0.936041, 0.094563],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 13 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: 3.140281895796733e-12,
#     2: 0.5799042634933687,
#     3: -0.015520611381581548,
#     4: -4.658694471813039,
#     5: 48.00775139689749,
#     6: -0.5789767818430653,
#     7: 0.3345754506634855,
#     8: 9.2759880860135,
# }
#
# FINAL SCORED SUBMISSION. One of eight matched the best value on record:
#   F2 0.579904 - new best, and the only function to finish on its peak.
#   F5 48.008 vs 566.342 banked in W1  (~11.8x lower)
#   F7 0.335  vs 2.424   banked in W2  (~7.2x lower)
#   F8 9.276  vs 9.644   banked in W2
#   F6 -0.579 vs -0.398  banked in W2
#   F1, F3, F4 also below their banked values.
# F4 and F7 both regressed on their first perpendicular probes, which is a clean
# negative: the line vertices were real optima in more than the line direction.
# F3's distant jump returned -0.015521 against -0.015429 - two widely separated
# points within 0.6% of each other, which reads as a broad shallow plateau rather
# than the narrow vertex I had diagnosed.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
